# Create Iceberg Tables for CDC Pipeline

This notebook creates the Iceberg tables needed for the CDC pipeline to work.

## Prerequisites
- Nessie running at http://nessie:19120
- MinIO running at http://minio:9000
- Network connectivity to both services

## Instructions
Run all cells in order from top to bottom.

In [ ]:
# Install required packages
import sys
import subprocess

packages = [
    "pyspark==3.5.0",
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ Packages installed")

## Step 1: Setup Spark with Iceberg

The next cell will:
- Download Iceberg Spark runtime libraries
- Download Nessie integration libraries
- Download AWS S3 libraries for MinIO
- Configure Spark session with Nessie catalog

**Note:** First run may take a few minutes to download dependencies (~200MB)

In [1]:
import os
from pyspark.sql import SparkSession

# Set AWS region (required by AWS SDK v2, even for MinIO)
os.environ['AWS_REGION'] = 'us-east-1'

# Package versions
packages = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0",
    "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1",
    "org.apache.iceberg:iceberg-aws-bundle:1.5.0"
]

packages_str = ",".join(packages)

# Set environment variables for Hadoop AWS
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {packages_str} pyspark-shell'

# Create Spark session with Iceberg and Nessie configuration
spark = SparkSession.builder \
    .appName("Create Iceberg CDC Tables") \
    .config("spark.jars.packages", packages_str) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://localhost:19120/api/v2") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.warehouse", "s3://lakehouse/warehouse") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://localhost:9000") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.sql.catalog.nessie.s3.region", "us-east-1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✓ Spark session created successfully with Iceberg support")

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lap16470/.ivy2/cache
The jars for the packages stored in: /Users/lap16470/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8e2430bd-0ffa-476c-956f-15aa6d36d371;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.77.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.5.0 in central
:: resolution report :: resolve 190ms :: artifacts dl 40ms
	:: modules in use:
	org.apache.iceberg#iceberg-aws-bundle;1.5.0 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_

✓ Spark session created successfully with Iceberg support


In [2]:
# Create namespace for CDC tables
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.taxi_cdc")
print("✓ Namespace 'taxi_cdc' created/verified")

✓ Namespace 'taxi_cdc' created/verified


## Step 2: Create Namespace and Tables

The following cells will create:
1. The `taxi_cdc` namespace in Nessie
2. Three tables: `trips`, `payment_types`, `vendors`
3. Each table includes CDC metadata fields (`__op`, `__source_ts_ms`, etc.)

In [3]:
# Create trips table
trips_ddl = """
CREATE TABLE IF NOT EXISTS nessie.taxi_cdc.trips (
    id BIGINT,
    vendor_id INT,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count INT,
    trip_distance DECIMAL(8,2),
    pickup_longitude DECIMAL(18,14),
    pickup_latitude DECIMAL(18,14),
    dropoff_longitude DECIMAL(18,14),
    dropoff_latitude DECIMAL(18,14),
    payment_type INT,
    fare_amount DECIMAL(8,2),
    extra DECIMAL(8,2),
    mta_tax DECIMAL(8,2),
    tip_amount DECIMAL(8,2),
    tolls_amount DECIMAL(8,2),
    total_amount DECIMAL(8,2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    __op STRING,
    __source_ts_ms BIGINT,
    __source_db STRING,
    __source_table STRING
) USING iceberg
PARTITIONED BY (days(pickup_datetime))
TBLPROPERTIES (
    'write.format.default' = 'parquet',
    'write.parquet.compression-codec' = 'snappy'
)
"""

spark.sql(trips_ddl)
print("✓ Trips table created successfully")

✓ Trips table created successfully


In [4]:
# Create payment_types table
payment_types_ddl = """
CREATE TABLE IF NOT EXISTS nessie.taxi_cdc.payment_types (
    id INT,
    name STRING,
    description STRING,
    __op STRING,
    __source_ts_ms BIGINT,
    __source_db STRING,
    __source_table STRING
) USING iceberg
TBLPROPERTIES (
    'write.format.default' = 'parquet',
    'write.parquet.compression-codec' = 'snappy'
)
"""

spark.sql(payment_types_ddl)
print("✓ Payment types table created successfully")

✓ Payment types table created successfully


25/12/29 21:49:53 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
# Create vendors table
vendors_ddl = """
CREATE TABLE IF NOT EXISTS nessie.taxi_cdc.vendors (
    id INT,
    name STRING,
    description STRING,
    __op STRING,
    __source_ts_ms BIGINT,
    __source_db STRING,
    __source_table STRING
) USING iceberg
TBLPROPERTIES (
    'write.format.default' = 'parquet',
    'write.parquet.compression-codec' = 'snappy'
)
"""

spark.sql(vendors_ddl)
print("✓ Vendors table created successfully")

✓ Vendors table created successfully


In [6]:
# Verify tables were created
print("Tables in taxi_cdc namespace:")
spark.sql("SHOW TABLES IN nessie.taxi_cdc").show(truncate=False)

Tables in taxi_cdc namespace:
+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|taxi_cdc |payment_types|false      |
|taxi_cdc |trips        |false      |
|taxi_cdc |vendors      |false      |
+---------+-------------+-----------+

+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|taxi_cdc |payment_types|false      |
|taxi_cdc |trips        |false      |
|taxi_cdc |vendors      |false      |
+---------+-------------+-----------+



## Step 3: Verify Tables

Check that all tables were created successfully:

In [ ]:
# Show schema of trips table
print("\nTrips table schema:")
spark.sql("DESCRIBE nessie.taxi_cdc.trips").show(50, truncate=False)

In [ ]:
# Test table access (should return empty result)
print("\nTest querying trips table:")
spark.sql("SELECT COUNT(*) as row_count FROM nessie.taxi_cdc.trips").show()

print("\n✅ All tables created successfully!")
print("You can now start the Iceberg Sink Connector.")

## Optional: Drop Tables (Use with caution!)

Run the cell below only if you need to recreate the tables from scratch.

In [ ]:
# Uncomment to drop tables
# spark.sql("DROP TABLE IF EXISTS nessie.taxi_cdc.trips")
# spark.sql("DROP TABLE IF EXISTS nessie.taxi_cdc.payment_types")
# spark.sql("DROP TABLE IF EXISTS nessie.taxi_cdc.vendors")
# print("Tables dropped")

## Step 4: Data Verification — Iceberg vs PostgreSQL

Cross-check row counts and field values between the source (PostgreSQL) and the Iceberg lakehouse tables populated by the CDC pipeline.

In [ ]:
import psycopg2
import pandas as pd

# ── PostgreSQL connection ───────────────────────────────────────────────────
pg = psycopg2.connect(
    host="localhost", port=5432,
    dbname="taxi_db", user="postgres", password="postgres"
)
cur = pg.cursor()

def pg_query(sql):
    cur.execute(sql)
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

# ── Row counts: PostgreSQL ──────────────────────────────────────────────────
pg_trips_count        = pg_query("SELECT COUNT(*) AS cnt FROM taxi.trips").iloc[0,0]
pg_payment_count      = pg_query("SELECT COUNT(*) AS cnt FROM taxi.payment_types").iloc[0,0]
pg_vendors_count      = pg_query("SELECT COUNT(*) AS cnt FROM taxi.vendors").iloc[0,0]

print(f"PostgreSQL row counts")
print(f"  trips:         {pg_trips_count:,}")
print(f"  payment_types: {pg_payment_count:,}")
print(f"  vendors:       {pg_vendors_count:,}")


In [ ]:
from pyspark.sql import functions as F

# ── Row counts: Iceberg ────────────────────────────────────────────────────
ice_trips_count   = spark.sql("SELECT COUNT(*) FROM nessie.taxi_cdc.trips").collect()[0][0]
ice_payment_count = spark.sql("SELECT COUNT(*) FROM nessie.taxi_cdc.payment_types").collect()[0][0]
ice_vendors_count = spark.sql("SELECT COUNT(*) FROM nessie.taxi_cdc.vendors").collect()[0][0]

print(f"Iceberg row counts")
print(f"  trips:         {ice_trips_count:,}")
print(f"  payment_types: {ice_payment_count:,}")
print(f"  vendors:       {ice_vendors_count:,}")
print()

# ── Count comparison ────────────────────────────────────────────────────────
rows = [
    ("trips",         pg_trips_count,   ice_trips_count),
    ("payment_types", pg_payment_count, ice_payment_count),
    ("vendors",       pg_vendors_count, ice_vendors_count),
]
print(f"{'Table':<16} {'PostgreSQL':>12} {'Iceberg':>12} {'Match?':>8}")
print("-" * 52)
all_match = True
for table, pg_cnt, ice_cnt in rows:
    match = "✓" if pg_cnt == ice_cnt else "✗ MISMATCH"
    if pg_cnt != ice_cnt:
        all_match = False
    print(f"{table:<16} {pg_cnt:>12,} {ice_cnt:>12,} {match:>8}")
print()
if all_match:
    print("✅ Row counts match across all tables")
else:
    print("⚠️  Row count mismatch — CDC lag or connector issue")


In [ ]:
# ── Field-level check: payment_types (small static table — full comparison) ─
pg_pt = pg_query("SELECT id, name, description FROM taxi.payment_types ORDER BY id")
ice_pt = spark.sql("""
    SELECT id, name, description
    FROM nessie.taxi_cdc.payment_types
    ORDER BY id
""").toPandas()

print("=== payment_types: PostgreSQL ===")
print(pg_pt.to_string(index=False))
print()
print("=== payment_types: Iceberg ===")
print(ice_pt.to_string(index=False))
print()

merged_pt = pg_pt.merge(ice_pt, on="id", suffixes=("_pg","_ice"))
mismatch_pt = merged_pt[merged_pt["name_pg"] != merged_pt["name_ice"]]
if mismatch_pt.empty:
    print("✅ payment_types: all field values match")
else:
    print("✗ Mismatches:")
    print(mismatch_pt)


In [ ]:
# ── Field-level check: vendors ──────────────────────────────────────────────
pg_v = pg_query("SELECT id, name, description FROM taxi.vendors ORDER BY id")
ice_v = spark.sql("SELECT id, name, description FROM nessie.taxi_cdc.vendors ORDER BY id").toPandas()

print("=== vendors: PostgreSQL ===")
print(pg_v.to_string(index=False))
print()
print("=== vendors: Iceberg ===")
print(ice_v.to_string(index=False))
print()

merged_v = pg_v.merge(ice_v, on="id", suffixes=("_pg","_ice"))
mismatch_v = merged_v[merged_v["name_pg"] != merged_v["name_ice"]]
print("✅ vendors: all field values match" if mismatch_v.empty else f"✗ Mismatches:\n{mismatch_v}")


In [ ]:
# ── Field-level check: trips — sample the first 10 by id ───────────────────
# Numeric fields to compare: fare_amount, total_amount, trip_distance, passenger_count
pg_trips = pg_query("""
    SELECT id, vendor_id, passenger_count,
           ROUND(trip_distance::numeric, 2)   AS trip_distance,
           ROUND(fare_amount::numeric, 2)      AS fare_amount,
           ROUND(total_amount::numeric, 2)     AS total_amount,
           payment_type
    FROM taxi.trips
    ORDER BY id
    LIMIT 10
""")

ice_trips = spark.sql("""
    SELECT id, vendor_id, passenger_count,
           ROUND(trip_distance, 2)  AS trip_distance,
           ROUND(fare_amount, 2)    AS fare_amount,
           ROUND(total_amount, 2)   AS total_amount,
           payment_type
    FROM nessie.taxi_cdc.trips
    ORDER BY id
    LIMIT 10
""").toPandas()

print("=== trips sample (first 10): PostgreSQL ===")
print(pg_trips.to_string(index=False))
print()
print("=== trips sample (first 10): Iceberg ===")
print(ice_trips.to_string(index=False))
print()

# Merge on id and compare numeric fields
numeric_cols = ["vendor_id", "passenger_count", "trip_distance", "fare_amount", "total_amount", "payment_type"]
merged_t = pg_trips.merge(ice_trips, on="id", suffixes=("_pg","_ice"))

mismatches = []
for col in numeric_cols:
    diff = merged_t[merged_t[f"{col}_pg"] != merged_t[f"{col}_ice"]]
    if not diff.empty:
        mismatches.append(f"  {col}: {len(diff)} row(s) differ")

if not mismatches:
    print("✅ trips sample: all numeric fields match exactly")
else:
    print("✗ trips sample mismatches:")
    for m in mismatches:
        print(m)


In [ ]:
# ── CDC metadata fields check ───────────────────────────────────────────────
print("=== CDC metadata fields on trips (latest 5 rows) ===")
spark.sql("""
    SELECT id, __op, __source_db, __source_table, __source_ts_ms
    FROM nessie.taxi_cdc.trips
    ORDER BY id DESC
    LIMIT 5
""").show(truncate=False)

# Verify __op values are only expected CDC operations
op_counts = spark.sql("""
    SELECT __op, COUNT(*) AS cnt
    FROM nessie.taxi_cdc.trips
    GROUP BY __op
    ORDER BY cnt DESC
""")
print("CDC operation distribution in trips:")
op_counts.show()

# ── Summary ─────────────────────────────────────────────────────────────────
print("=" * 52)
print("VERIFICATION SUMMARY")
print("=" * 52)
print(f"  PostgreSQL trips:         {pg_trips_count:>8,}")
print(f"  Iceberg trips:            {ice_trips_count:>8,}")
print(f"  PostgreSQL payment_types: {pg_payment_count:>8,}")
print(f"  Iceberg payment_types:    {ice_payment_count:>8,}")
print(f"  PostgreSQL vendors:       {pg_vendors_count:>8,}")
print(f"  Iceberg vendors:          {ice_vendors_count:>8,}")

pg.close()
